# Quarantine Retention Cleanup

Manage retention and cleanup of quarantine records to prevent unbounded growth.

**Purpose**: Archive and purge old quarantine records based on configurable retention policies

**Retention Policies**:
- **REPROCESSED**: Archive after 30 days, delete after 90 days
- **DISCARDED**: Archive after 7 days, delete after 30 days
- **PENDING**: Alert if older than 60 days (requires review)
- **REPROCESS_FAILED**: Keep for investigation, archive after 60 days

**Workflow**:
1. Identify records meeting retention criteria
2. Archive to historical table (optional)
3. Delete from active quarantine
4. Log cleanup metrics

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, current_timestamp, datediff, lit, count
)
from datetime import datetime, timedelta

# Configuration
QUARANTINE_TABLE = "workspace.quarantine.quarantine_jobs"
QUARANTINE_ARCHIVE = "workspace.quarantine.quarantine_jobs_archive"

# Retention periods (in days)
RETENTION_POLICIES = {
    "REPROCESSED": {
        "archive_after_days": 30,
        "delete_after_days": 90,
        "description": "Successfully reprocessed records"
    },
    "DISCARDED": {
        "archive_after_days": 7,
        "delete_after_days": 30,
        "description": "Permanently discarded records"
    },
    "PENDING": {
        "alert_after_days": 60,
        "archive_after_days": 180,
        "delete_after_days": 365,
        "description": "Unresolved records requiring attention"
    },
    "REPROCESS_FAILED": {
        "archive_after_days": 60,
        "delete_after_days": 180,
        "description": "Failed reprocessing attempts"
    }
}

print("Quarantine Retention Policies:")
print("=" * 80)
for status, policy in RETENTION_POLICIES.items():
    print(f"\n{status}:")
    print(f"  {policy['description']}")
    for key, val in policy.items():
        if key != 'description':
            print(f"  - {key}: {val} days")

In [0]:
def analyze_retention_status():
    """
    Analyze current quarantine records against retention policies.
    """
    df = spark.table(QUARANTINE_TABLE)
    
    print("\n" + "=" * 80)
    print("QUARANTINE RETENTION ANALYSIS")
    print("=" * 80)
    
    # Overall age distribution
    print("\n📅 Age Distribution by Status:\n")
    
    age_df = df.withColumn(
        "age_days",
        datediff(current_timestamp(), col("quarantined_at"))
    ).withColumn(
        "age_bucket",
        F.when(col("age_days") < 7, "0-7 days")
        .when(col("age_days") < 30, "8-30 days")
        .when(col("age_days") < 60, "31-60 days")
        .when(col("age_days") < 90, "61-90 days")
        .when(col("age_days") < 180, "91-180 days")
        .otherwise("180+ days")
    )
    
    age_df.groupBy("reprocess_status", "age_bucket").count() \
        .orderBy("reprocess_status", "age_bucket").show(50, truncate=False)
    
    # Records eligible for archive/delete
    print("\n⚠️  Records Exceeding Retention Thresholds:\n")
    
    for status, policy in RETENTION_POLICIES.items():
        status_df = df.filter(col("reprocess_status") == status)
        total_count = status_df.count()
        
        if total_count == 0:
            continue
        
        print(f"\n{status} ({total_count} records):")
        
        # Check archive threshold
        if "archive_after_days" in policy:
            archive_days = policy["archive_after_days"]
            archive_df = status_df.filter(
                datediff(current_timestamp(), col("quarantined_at")) > archive_days
            )
            archive_count = archive_df.count()
            if archive_count > 0:
                print(f"  📦 Archive eligible (>{archive_days}d): {archive_count}")
        
        # Check delete threshold
        if "delete_after_days" in policy:
            delete_days = policy["delete_after_days"]
            delete_df = status_df.filter(
                datediff(current_timestamp(), col("quarantined_at")) > delete_days
            )
            delete_count = delete_df.count()
            if delete_count > 0:
                print(f"  🗑️  Delete eligible (>{delete_days}d): {delete_count}")
        
        # Check alert threshold for PENDING
        if "alert_after_days" in policy:
            alert_days = policy["alert_after_days"]
            alert_df = status_df.filter(
                datediff(current_timestamp(), col("quarantined_at")) > alert_days
            )
            alert_count = alert_df.count()
            if alert_count > 0:
                print(f"  ⚠️  Alert (>{alert_days}d, needs review): {alert_count}")

# Run analysis
analyze_retention_status()

In [0]:
def archive_records(
    status: str,
    archive_after_days: int,
    dry_run: bool = True
) -> int:
    """
    Archive records to historical table.
    
    Args:
        status: Reprocess status to archive
        archive_after_days: Archive records older than this many days
        dry_run: If True, only show what would be archived
        
    Returns:
        Count of records archived
    """
    df = spark.table(QUARANTINE_TABLE)
    
    # Find records to archive
    archive_df = df.filter(
        (col("reprocess_status") == status) &
        (datediff(current_timestamp(), col("quarantined_at")) > archive_after_days)
    )
    
    count = archive_df.count()
    
    if count == 0:
        print(f"\nℹ️  No {status} records to archive (>{archive_after_days} days)")
        return 0
    
    print(f"\n📦 Found {count} {status} records to archive (>{archive_after_days} days)")
    
    if dry_run:
        print("🔍 DRY RUN - No actual archiving")
        archive_df.select(
            "quarantine_id", "source_name", "quarantined_at", "reprocess_status"
        ).show(10)
        return count
    
    # Create archive table if it doesn't exist
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {QUARANTINE_ARCHIVE}
    LIKE {QUARANTINE_TABLE}
    """)
    
    # Add archive metadata
    archive_df = archive_df.withColumn(
        "archived_at", current_timestamp()
    )
    
    # Write to archive
    archive_df.write.mode("append").saveAsTable(QUARANTINE_ARCHIVE)
    
    print(f"✓ Archived {count} records to {QUARANTINE_ARCHIVE}")
    
    return count

In [0]:
def delete_archived_records(
    status: str,
    delete_after_days: int,
    dry_run: bool = True
) -> int:
    """
    Delete records that have been archived and exceeded retention.
    
    Args:
        status: Reprocess status to delete
        delete_after_days: Delete records older than this many days
        dry_run: If True, only show what would be deleted
        
    Returns:
        Count of records deleted
    """
    df = spark.table(QUARANTINE_TABLE)
    
    # Find records to delete
    delete_df = df.filter(
        (col("reprocess_status") == status) &
        (datediff(current_timestamp(), col("quarantined_at")) > delete_after_days)
    )
    
    count = delete_df.count()
    
    if count == 0:
        print(f"\nℹ️  No {status} records to delete (>{delete_after_days} days)")
        return 0
    
    print(f"\n🗑️  Found {count} {status} records to delete (>{delete_after_days} days)")
    
    if dry_run:
        print("🔍 DRY RUN - No actual deletion")
        delete_df.select(
            "quarantine_id", "source_name", "quarantined_at", "reprocess_status"
        ).show(10)
        return count
    
    # Collect IDs to delete
    ids_to_delete = [row["quarantine_id"] for row in delete_df.collect()]
    
    if ids_to_delete:
        ids_str = "', '".join(ids_to_delete)
        delete_sql = f"""
        DELETE FROM {QUARANTINE_TABLE}
        WHERE quarantine_id IN ('{ids_str}')
        """
        spark.sql(delete_sql)
        
        print(f"✓ Deleted {count} records from {QUARANTINE_TABLE}")
    
    return count

In [0]:
def alert_stale_pending(alert_after_days: int = 60):
    """
    Generate alerts for PENDING records that need attention.
    
    Args:
        alert_after_days: Alert on records pending longer than this
    """
    df = spark.table(QUARANTINE_TABLE)
    
    stale_df = df.filter(
        (col("reprocess_status") == "PENDING") &
        (datediff(current_timestamp(), col("quarantined_at")) > alert_after_days)
    )
    
    count = stale_df.count()
    
    print("\n" + "=" * 80)
    print(f"⚠️  STALE PENDING RECORDS ALERT (>{alert_after_days} days)")
    print("=" * 80)
    
    if count == 0:
        print(f"\n✅ No stale PENDING records found")
        return
    
    print(f"\n⚠️  Found {count} PENDING records requiring review\n")
    
    # Group by source and rule
    print("By Source and Failed Rule:")
    stale_df.groupBy("source_name", "failed_rule_name").count() \
        .orderBy("count", ascending=False).show(20, truncate=False)
    
    # Show oldest records
    print("\nOldest PENDING Records:")
    stale_df.select(
        "quarantine_id",
        "source_name",
        "failed_rule_name",
        "quarantined_at",
        datediff(current_timestamp(), col("quarantined_at")).alias("days_pending")
    ).orderBy("quarantined_at").show(10, truncate=False)
    
    print("\n📌 ACTION REQUIRED: Review and resolve these records using quarantine_review_apply notebook")

# Run alert check
alert_stale_pending(alert_after_days=60)

In [0]:
def run_cleanup(
    dry_run: bool = True,
    enable_archive: bool = True,
    enable_delete: bool = False
) -> dict:
    """
    Run full cleanup process based on retention policies.
    
    Args:
        dry_run: If True, only show what would be done
        enable_archive: Enable archiving step
        enable_delete: Enable deletion step (requires archive to be disabled in dry run)
        
    Returns:
        Dictionary with cleanup results
    """
    results = {
        "archived": 0,
        "deleted": 0,
        "alerted": 0
    }
    
    print("\n" + "=" * 80)
    print("QUARANTINE RETENTION CLEANUP")
    print(f"Mode: {'DRY RUN' if dry_run else 'LIVE'}")
    print("=" * 80)
    
    # Process each status based on retention policy
    for status, policy in RETENTION_POLICIES.items():
        print(f"\n{'=' * 80}")
        print(f"Processing: {status}")
        print(f"{policy['description']}")
        print(f"{'=' * 80}")
        
        # Archive if enabled
        if enable_archive and "archive_after_days" in policy:
            archived = archive_records(
                status=status,
                archive_after_days=policy["archive_after_days"],
                dry_run=dry_run
            )
            results["archived"] += archived
        
        # Delete if enabled (only after archiving)
        if enable_delete and "delete_after_days" in policy:
            deleted = delete_archived_records(
                status=status,
                delete_after_days=policy["delete_after_days"],
                dry_run=dry_run
            )
            results["deleted"] += deleted
    
    # Always check for stale PENDING records
    print("\n" + "=" * 80)
    alert_stale_pending(alert_after_days=RETENTION_POLICIES["PENDING"]["alert_after_days"])
    
    print("\n" + "=" * 80)
    print("CLEANUP SUMMARY")
    print("=" * 80)
    print(f"  📦 Records archived: {results['archived']}")
    print(f"  🗑️  Records deleted: {results['deleted']}")
    print("=" * 80)
    
    if dry_run:
        print("\n⚠️  This was a DRY RUN - no changes made")
        print("Set dry_run=False to execute actual cleanup")
    
    return results

In [0]:
# First, analyze current retention status
analyze_retention_status()

# Run dry run to preview cleanup actions
print("\n" + "=" * 80)
print("RUNNING CLEANUP DRY RUN")
print("=" * 80)

results = run_cleanup(
    dry_run=True,
    enable_archive=True,
    enable_delete=True
)

print(f"\n📊 Cleanup Preview Results:")
print(f"  - Would archive: {results['archived']} records")
print(f"  - Would delete: {results['deleted']} records")

# Uncomment to run actual cleanup
# CAUTION: This will permanently modify data
# 
# live_results = run_cleanup(
#     dry_run=False,
#     enable_archive=True,
#     enable_delete=True  # Only enable after archiving is confirmed working
# )

In [0]:
def manual_cleanup_by_date(
    status: str,
    before_date: str,
    dry_run: bool = True
) -> int:
    """
    Manually delete records by status and date (emergency cleanup).
    
    Args:
        status: Reprocess status
        before_date: Delete records quarantined before this date (YYYY-MM-DD)
        dry_run: If True, preview only
        
    Returns:
        Count of records affected
    """
    df = spark.table(QUARANTINE_TABLE)
    
    target_df = df.filter(
        (col("reprocess_status") == status) &
        (col("quarantined_at") < before_date)
    )
    
    count = target_df.count()
    
    print(f"\n🗑️  Manual Cleanup: {status} records before {before_date}")
    print(f"Found {count} records")
    
    if count == 0:
        return 0
    
    if dry_run:
        print("🔍 DRY RUN")
        target_df.groupBy("source_name").count().show()
        return count
    
    # Execute deletion
    ids = [row["quarantine_id"] for row in target_df.collect()]
    ids_str = "', '".join(ids)
    
    delete_sql = f"""
    DELETE FROM {QUARANTINE_TABLE}
    WHERE quarantine_id IN ('{ids_str}')
    """
    spark.sql(delete_sql)
    
    print(f"✓ Deleted {count} records")
    return count

# Example: Delete all REPROCESSED records before 2024-01-01
# manual_cleanup_by_date(
#     status="REPROCESSED",
#     before_date="2024-01-01",
#     dry_run=True
# )